# Наличие товара на филиалах по дням

Задача: для одной товарной категории построить полную историю **остатков на складе каждого филиала по
каждому дню** - хотя система не хранит ежедневный снимок остатков, только текущий остаток и журнал движений
(поступления/продажи/списания). Ежедневный снимок восстанавливается расчётом: берём текущий остаток и
"откатываем" его назад по дням, вычитая движения - вместо того чтобы запрашивать историю остатков напрямую
(что для СУБД намного тяжелее, чем выгрузить текущий остаток + журнал движений).

Итог - таблица Дата x Товар x Филиал с восстановленным остатком (в штуках и в рублях по приоритетной цене)
и фактическими продажами за день. На этой таблице дальше строится анализ наличия/дефицита товара на полке
конкретного филиала во времени - а не только "на сегодня".

Для портфолио: подключение к БД - функция-заглушка, конкретная категория и диапазон дат - лишь пример
запуска. Запросы обращаются к типовым объектам метаданных 1С, а не к специфичной для конкретной компании схеме.

## Импорт библиотек

In [ ]:
import pandas as pd
import numpy as np
import itertools
import gc
from datetime import datetime
from tqdm import tqdm

pd.options.display.max_columns = 100

## Настройка параметров

In [ ]:
# Категория, для которой строится расчёт (первый элемент - для подстановки в SQL-кортеж)
CATEGORY = 'Apple Смартфоны'
CATEGORY_SQL_TUPLE = (CATEGORY, 'ДляКортежа')

# Период: с 1-го числа текущего месяца по сегодня
START_DATE = datetime.today().replace(day=1)
END_DATE = datetime.today()

# Виды филиалов, которые учитываются в расчёте (обычные магазины, дисконт-центры, РРЦ - не склады/офисы)
RETAIL_BRANCH_TYPES = ['Магазин', 'Дисконт центр', 'РРЦ']

## Функции

In [ ]:
def get_db_engine():
    """Заглушка вместо реального подключения к 1С-базе (в рабочей версии - через внутренний пакет доступа к БД)."""
    raise NotImplementedError("Подключение к реальной БД в портфолио-версии не используется")

In [ ]:
def reduce_mem_usage(df: pd.DataFrame, int_cast=True, obj_to_category=False, subset=None) -> pd.DataFrame:
    """
    Понижает типы данных датафрейма до минимально достаточных (int8/16/32, float16/32) - на таблицах вида
    Дата x Товар x Филиал (миллионы строк) это ощутимо снижает потребление памяти.
    """
    start_mem = df.memory_usage().sum() / 1024 ** 2
    gc.collect()
    print('Memory usage of dataframe is {:.2f} MB'.format(start_mem))

    cols = subset if subset is not None else df.columns.tolist()
    for col in tqdm(cols):
        col_type = df[col].dtype
        if col_type != object and col_type.name != 'category' and 'datetime' not in col_type.name:
            c_min, c_max = df[col].min(), df[col].max()
            treat_as_int = str(col_type)[:3] == 'int' or (int_cast and isinstance(df[col], int))
            if treat_as_int:
                for np_type in (np.int8, np.uint8, np.int16, np.uint16, np.int32, np.uint32, np.int64, np.uint64):
                    info = np.iinfo(np_type)
                    if c_min > info.min and c_max < info.max:
                        df[col] = df[col].astype(np_type)
                        break
            else:
                for np_type in (np.float16, np.float32):
                    info = np.finfo(np_type)
                    if c_min > info.min and c_max < info.max:
                        df[col] = df[col].astype(np_type)
                        break
                else:
                    df[col] = df[col].astype(np.float64)
        elif 'datetime' not in col_type.name and obj_to_category:
            df[col] = df[col].astype('category')

    gc.collect()
    end_mem = df.memory_usage().sum() / 1024 ** 2
    print('Memory usage after optimization is: {:.3f} MB'.format(end_mem))
    print('Decreased by {:.1f}%'.format(100 * (start_mem - end_mem) / start_mem))
    return df

## Алгоритм выполнения

### Календарь и справочник категорий

In [ ]:
engine = get_db_engine()

calendar_df = pd.DataFrame({'datetime': pd.date_range(start=START_DATE, end=END_DATE, freq='D')})
calendar_df['Дата'] = calendar_df['datetime'].dt.date.astype(str)
calendar_df = calendar_df.reset_index().rename(columns={'index': 'i_date'})
reduce_mem_usage(calendar_df).info()

Иерархия категорий + справочник товаров выбранной категории, с ассортиментным статусом товара
(готовый статус из 1С - класс оборачиваемости/приоритет закупки, используется здесь просто как атрибут
для последующих срезов, не пересчитывается).

In [ ]:
category_hierarchy_query = f"""
-- иерархия категорий (та же структура, что в 01_справочник_автосегментов.ipynb),
-- отфильтрованная до {CATEGORY_SQL_TUPLE} на 4-м уровне, с джойном на справочник номенклатуры
-- и присоединённым "Ассортиментный_Статус" (готовый статус товара из 1С: класс/приоритет закупки)
"""
df_category = engine.execute_to_df(category_hierarchy_query + '\n SELECT * FROM #ТоварВИерархии')
df_category = df_category.reset_index().rename(columns={'index': 'i_product'})
reduce_mem_usage(df_category).info()

### Приоритетная цена товара

Из нескольких типов цен в 1С (ФЦ ОРП, РФЦ обычная, РФЦ от себестоимости) берётся первая заполненная по
приоритету - используется дальше только для перевода остатков/продаж из штук в рубли.

In [ ]:
price_query = """
-- те же товары категории, приоритетная цена = coalesce(ФЦ_ОРП, РФЦ_обычная, РФЦ_от_себестоимости, 0)
"""
df_price = engine.execute_to_df(price_query)
df_price = df_price.merge(df_category[['ТоварСсылка', 'i_product']], on='ТоварСсылка').drop(columns='ТоварСсылка')
df_price['ПриоритетнаяЦена'] = df_price['ПриоритетнаяЦена'].astype(int)
reduce_mem_usage(df_price).info()

### Справочник филиалов

In [ ]:
branches_query = """
-- справочник филиалов: ссылка, название, вид филиала (магазин/склад/дисконт-центр/РРЦ), привязка к складу
"""
df_branches = engine.execute_to_df(branches_query)
df_branches = df_branches.reset_index().rename(columns={'index': 'i_branch'})
reduce_mem_usage(df_branches).info()

### Движения товара (счёт 41) и продажи (счёт 90)

"Счёт 41" и "Счёт 90" - стандартные бухгалтерские регистры 1С: 41 - движение товара на складе
(поступления/продажи/списания в штуках и сумме), 90 - реализация (фактические продажи). Оба берутся с начала
периода `START_DATE` и дальше агрегируются по дню/товару/филиалу.

In [ ]:
schet_41_query = f"""
-- движения по счёту 41 для товаров категории с {START_DATE.date()}, агрегированные по
-- (Дата, ВидДвижения, Номенклатура, Филиал): сумма количества и суммы без НДС
"""
df_schet_41 = engine.execute_to_df(schet_41_query)
df_schet_41['Дата'] = df_schet_41['Дата'].astype(str)

df_schet_41 = df_schet_41.merge(df_category[['ТоварСсылка', 'i_product']], on='ТоварСсылка')
df_schet_41 = df_schet_41.merge(df_branches[['ФилиалСсылка', 'i_branch']], on='ФилиалСсылка')
df_schet_41 = df_schet_41.merge(calendar_df[['Дата', 'i_date']], on='Дата')
df_schet_41 = df_schet_41.drop(columns=['ТоварСсылка', 'ФилиалСсылка', 'Дата'])
df_schet_41[['ВидДвижения', 'Количество', 'Сумма']] = df_schet_41[['ВидДвижения', 'Количество', 'Сумма']].astype(int)
reduce_mem_usage(df_schet_41).info()

In [ ]:
schet_90_query = """
-- реализация (продажи) по счёту 90 для товаров категории - по (Дата, Номенклатура, Филиал):
-- сумма количества и себестоимости
"""
df_schet_90 = engine.execute_to_df(schet_90_query)
df_schet_90['Дата'] = df_schet_90['Дата'].astype(str)

df_schet_90 = df_schet_90.merge(df_category[['ТоварСсылка', 'i_product']], on='ТоварСсылка')
df_schet_90 = df_schet_90.merge(df_branches[['ФилиалСсылка', 'i_branch']], on='ФилиалСсылка')
df_schet_90 = df_schet_90.merge(calendar_df[['Дата', 'i_date']], on='Дата')
df_schet_90 = df_schet_90.drop(columns=['ТоварСсылка', 'ФилиалСсылка', 'Дата'])
df_schet_90[['СуммаСебес', 'Количество']] = df_schet_90[['СуммаСебес', 'Количество']].astype(int)
reduce_mem_usage(df_schet_90).info()

### Текущий остаток

In [ ]:
stock_query = """
-- текущий остаток (на сегодня) по товарам категории и филиалам: Остаток, Резерв, МягкийРезерв, Транзит, Путь
"""
df_stock_now = engine.execute_to_df(stock_query)
df_stock_now['Дата'] = datetime.today().date().strftime('%Y-%m-%d')

df_stock_now = df_stock_now.merge(df_category[['ТоварСсылка', 'i_product']], on='ТоварСсылка')
df_stock_now = df_stock_now.merge(df_branches[['ФилиалСсылка', 'i_branch']], on='ФилиалСсылка')
df_stock_now = df_stock_now.merge(calendar_df[['Дата', 'i_date']], on='Дата')
df_stock_now = df_stock_now.drop(columns=['ТоварСсылка', 'ФилиалСсылка', 'Дата'])
df_stock_now[['Остаток', 'Резерв', 'МягкийРезерв', 'Транзит', 'Путь']] = \
    df_stock_now[['Остаток', 'Резерв', 'МягкийРезерв', 'Транзит', 'Путь']].astype(int)
reduce_mem_usage(df_stock_now).info()

### Восстановление дневных остатков

Знак движения зависит от его вида (поступление увеличивает остаток, продажа/списание уменьшает) - здесь
это сворачивается в один знак-множитель, дальше движения агрегируются по дню/товару/филиалу.

In [ ]:
df_schet_41['Множитель'] = df_schet_41['ВидДвижения'].apply(lambda x: -1 if x == 1 else 1)
df_schet_41['Движения'] = df_schet_41['Количество'] * df_schet_41['Множитель']
df_schet_41['Сумма_мод'] = df_schet_41['Сумма'] * df_schet_41['Множитель']

schet_41_by_day = df_schet_41[['i_date', 'i_product', 'i_branch', 'Движения', 'Сумма_мод']]\
    .groupby(by=['i_date', 'i_product', 'i_branch']).agg('sum').reset_index()

Полная матрица Дата x Товар x Филиал (декартово произведение) - нужна, чтобы у каждого товара на каждом
филиале была строка на каждый день периода, даже если в этот день не было ни движений, ни продаж (иначе
восстановление остатка "назад по дням" ниже даст разрывы).

In [ ]:
retail_branches = df_branches[df_branches['Вид_филиала'].isin(RETAIL_BRANCH_TYPES)]
products_with_movements = list(set(schet_41_by_day['i_product']))

matrix = pd.DataFrame(
    list(itertools.product(calendar_df['i_date'], retail_branches['i_branch'], pd.Series(products_with_movements))),
    columns=['i_date', 'i_branch', 'i_product'],
)
reduce_mem_usage(matrix).info()

**Восстановление остатка по дням.** Идея: текущий остаток известен точно (последний день периода), а
дальше движемся назад по датам и вычитаем чистое движение за день - получаем остаток "на начало дня" для
каждого более раннего дня. Реализовано через `cumsum()` по датам в обратном порядке (сортировка по убыванию
даты + накопительная сумма разницы "остаток - движение" = восстановленный остаток на этот день).

In [ ]:
merged = matrix.merge(df_stock_now[['i_date', 'i_branch', 'i_product', 'Остаток']], on=['i_date', 'i_branch', 'i_product'], how='left')
merged = merged.merge(schet_41_by_day[['i_date', 'i_branch', 'i_product', 'Движения']], on=['i_date', 'i_branch', 'i_product'], how='left')
merged = merged.merge(df_schet_90[['i_date', 'i_branch', 'i_product', 'Количество']], on=['i_date', 'i_branch', 'i_product'], how='left')
reduce_mem_usage(merged).info()

merged = merged.fillna(0)
merged['ПоДням'] = merged['Остаток'] - merged['Движения']

merged = merged.sort_values(by=['i_product', 'i_branch', 'i_date'], ascending=False)
merged['Остатки_шт'] = merged.groupby(['i_product', 'i_branch'])['ПоДням'].cumsum()

# строки, где восстановленный остаток за всю историю так и остался нулевым - товар не появлялся на филиале, выбрасываем
merged = merged[merged['Остатки_шт'] != 0]

### Сборка итоговой таблицы

In [ ]:
merged = merged.rename(columns={'Количество': 'Продажи_шт'})

merged = merged.merge(df_category[['i_product', 'Код', 'Товар', 'Ассортиментный_Статус']], on='i_product', how='left')
merged = merged.merge(df_branches[['i_branch', 'Филиал', 'Вид_филиала', 'ПриписанК_Складу']], on='i_branch', how='left')
merged = merged.merge(calendar_df[['i_date', 'Дата']], on='i_date', how='left')
merged = merged.merge(df_price[['i_product', 'ПриоритетнаяЦена']], on='i_product', how='left')
merged = merged.drop(columns=['i_date', 'i_branch', 'i_product', 'Остаток', 'ПоДням'])

merged['Остатки_руб'] = merged['Остатки_шт'] * merged['ПриоритетнаяЦена']
merged['Продажи_руб'] = merged['Продажи_шт'] * merged['ПриоритетнаяЦена']
merged